# Tutorial 2: Redis Data Structures (Lecture 12)

In Tutorial 1 every value we stored was a single string. If that were all Redis offered, it would be a dictionary that happens to live on the network.

What makes it useful is that a value can itself be a data structure, and that Redis understands the structure rather than treating it as an opaque blob. The operations then happen on the server: adding one item to a list of a million does not mean fetching the list, changing it in Python, and writing the whole thing back.

This notebook works through the five structures, then key expiry, and ends with the pattern that is the most common reason a team runs Redis at all. Each structure is built from the same orders dataset used in the MySQL tutorials, so you can compare directly against the tables you already know.

In [1]:
import csv
import redis

REDIS_HOST = "redis"      # use "localhost" if running outside the jupyter container
REDIS_PORT = 6379

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, decode_responses=True)
r.ping()

True

We will be reading the item and order data a few times over, so load the CSVs once here.

In [3]:
HOME_DIR="/home/jovyan/iem4723/mysql-dev/data/orders"

def load(name):
    with open(f"{HOME_DIR}/{name}", newline="") as f:
        return list(csv.DictReader(f))

items      = load("Item.csv")
customers  = load("Customer.csv")
orders     = load("CustomerOrder.csv")
order_item = load("Order_Item.csv")

print(len(items), "items,", len(customers), "customers,", len(orders), "orders")
items[:2]

20 items, 10 customers, 9 orders


[{'Item_ID': '121312-0026',
  'Item_Description': 'Slacks',
  'Item_Price': '13',
  'Item_Weight': '0',
  'Item_AnnualSales': '60',
  'Item_StockLocation': 'J6699'},
 {'Item_ID': '121332-0036',
  'Item_Description': 'Slacks',
  'Item_Price': '10',
  'Item_Weight': '0',
  'Item_AnnualSales': '2704',
  'Item_StockLocation': 'J6699'}]

## 1. Strings That Count

The simplest value, and the one from Tutorial 1. What makes it more interesting than it first looks is that Redis knows when a string holds a number.

In [4]:
r.set("page:views", 0)

print(r.incr("page:views"))        # 1
print(r.incr("page:views"))        # 2
print(r.incrby("page:views", 10))  # 12
print(r.decr("page:views"))        # 11

print("stored as:", repr(r.get("page:views")))

1
2
12
11
stored as: '11'


Notice the last line. The value comes back as the *string* `'11'`, not the integer `11`. Redis does the arithmetic on the server but everything in the keyspace is still text, so remember the `int()` when you need to compute with what you read back.

Each of these calls is **atomic**. Two clients incrementing at the same instant both count. That is precisely the thing a read, add one, write back sequence gets wrong under concurrency:

```python
count = int(r.get("page:views"))   # another client can slip in
r.set("page:views", count + 1)     # right here, and its increment is lost
```

Doing this safely in SQL means a transaction and a row lock. Here it is one call.

In [5]:
# a counter per order, built from the real order lines
for key in r.keys("order:*:lines"):
    r.delete(key)                      # start from zero if this cell is re-run

for line in order_item:
    r.incr(f"order:{line['Order_Num']}:lines")

for order in orders[:4]:
    num = order["Order_Num"]
    print(num, "->", r.get(f"order:{num}:lines"), "lines")

16639375 -> 3 lines
16642826 -> 4 lines
16644074 -> 2 lines
16645282 -> 4 lines


## 2. Hashes

A hash stores field and value pairs under a single key, which makes it the closest thing Redis has to a row in a relational table.

In Tutorial 1 we gave each customer field its own key: `customer:C25:city`, `customer:C25:state`, and so on. A hash keeps the whole record together under one key instead.

In [ ]:
r.hset("customer:C25", mapping={
    "first_name": "Bob",
    "last_name":  "Felton",
    "city":       "HERSHEY",
})

print(r.hget("customer:C25", "city"))
print(r.hgetall("customer:C25"))
print(r.hkeys("customer:C25"))

`hgetall` hands back an ordinary Python dict, which is usually the form you want.

Now the same thing for every customer in the file. One key each, every field in the hash.

In [ ]:
for row in customers:
    r.hset(f"customer:{row['Customer_Num']}", mapping={
        "first_name": row["Customer_FirstName"],
        "last_name":  row["Customer_LastName"],
        "city":       row["Customer_City"],
        "state":      row["Customer_State"],
        "email":      row["Customer_Email"],
    })

print(r.hgetall("customer:C2501"))
print()
print("just the city, without fetching the rest:", r.hget("customer:C2501", "city"))

Compare this to the `customer` table from the MySQL work. The same record, but with no schema declared ahead of time and no requirement that the next customer carry the same fields.

That freedom is also the cost. Nothing stops a typo creating a field named `citty`, and Redis will store it without complaint. A relational schema would have rejected it outright.

In [ ]:
r.hset("customer:C25", "citty", "HERSHEY")    # a typo Redis is happy to keep
print(r.hgetall("customer:C25"))

r.hdel("customer:C25", "citty")               # no constraint caught this, we did
print(r.hgetall("customer:C25"))

## 3. Lists

A list holds an ordered sequence of strings and can be pushed or popped at either end. `rpush` appends on the right, `lpush` on the left.

In [ ]:
r.delete("orders:pending")

r.rpush("orders:pending", "16642826", "16644074")
r.lpush("orders:pending", "16639375")          # goes on the front

print(r.lrange("orders:pending", 0, -1))       # every element
print("length:", r.llen("orders:pending"))

Index `-1` means the last element, exactly as in Python, so `0, -1` is the whole list.

Pushing at one end and popping at the other gives a **queue**, which is how Redis lists are most often used in practice: job queues, work backlogs, anything processed in order.

In [ ]:
r.delete("orders:to_ship")

for order in orders:
    r.rpush("orders:to_ship", order["Order_Num"])     # arrive at the back

print("queued:", r.llen("orders:to_ship"))

while True:
    order_num = r.lpop("orders:to_ship")              # leave from the front
    if order_num is None:
        break
    print("shipping", order_num)

print("queue empty:", r.llen("orders:to_ship") == 0)

`lpop` returning `None` on an empty list is what ends that loop, and it is the normal way to drain a queue.

Lists allow duplicates and keep insertion order, which is exactly what a queue needs and exactly what the next structure refuses to do.

## 4. Sets

A set is an unordered collection with no duplicates. The question it answers quickly is whether something is a member.

In [ ]:
r.delete("items:shirts", "items:location:A9309")

r.sadd("items:shirts", "14", "18", "25", "38", "71")
r.sadd("items:location:A9309", "14", "18", "25", "38", "71")

print("is 25 a shirt?  ", r.sismember("items:shirts", "25"))
print("how many shirts:", r.scard("items:shirts"))
print("the whole set:  ", sorted(r.smembers("items:shirts")))

Note that `sismember` answers `1`, not `True`. It is truthy, so `if r.sismember(...)` reads naturally, but printing it looks surprising the first time.

Adding a member that is already there changes nothing, so uniqueness is free rather than something to enforce. The return value is how many members were *actually new*.

In [ ]:
print("adding 25 again, newly added:", r.sadd("items:shirts", "25"))
print("adding 99 for the first time:", r.sadd("items:shirts", "99"))
print("size now:", r.scard("items:shirts"))

r.srem("items:shirts", "99")
print("after removing 99:", r.scard("items:shirts"))

Now build the sets properly, from the item file: one set per description, one set per stock location.

In [ ]:
for key in r.keys("items:description:*") + r.keys("items:location:*"):
    r.delete(key)

for row in items:
    r.sadd(f"items:description:{row['Item_Description']}", row["Item_ID"])
    r.sadd(f"items:location:{row['Item_StockLocation']}", row["Item_ID"])

print("descriptions:", sorted(k.split(":")[-1] for k in r.keys("items:description:*")))
print("shirts       :", sorted(r.smembers("items:description:Shirts")))
print("in A9309     :", sorted(r.smembers("items:location:A9309")))

The set operations run on the server, which is where this stops being a convenience and starts being the point. `sinter` intersects, `sunion` combines, `sdiff` subtracts.

That first question below, *which shirts are stocked in A9309*, is the kind of question that needs a join in SQL.

In [ ]:
print("shirts stocked in A9309:",
      sorted(r.sinter("items:description:Shirts", "items:location:A9309")))

print("everything in A9309 or L3234:",
      sorted(r.sunion("items:location:A9309", "items:location:L3234")))

print("slacks NOT in L3234:",
      sorted(r.sdiff("items:description:Slacks", "items:location:L3234")))

## 5. Sorted Sets

A sorted set is a set where every member also carries a numeric **score**, and Redis keeps the members ordered by it.

Watch the argument order here. In redis-py the members and scores go in as a dictionary, `{member: score}`, which is the reverse of the `ZADD key score member` form used by the command line and by the textbook.

In [ ]:
r.delete("items:by_sales")

r.zadd("items:by_sales", {
    "71": 6672, "121332-0036": 2704, "25": 1016,
})

print(r.zrevrange("items:by_sales", 0, 2, withscores=True))
print("score of 71:", r.zscore("items:by_sales", "71"))
print("rank of 25 :", r.zrank("items:by_sales", "25"), "(lowest first, 0-based)")

Scores come back as floats, even where they went in as integers.

Now the whole catalogue, scored by annual sales. The ordering is maintained as each item is inserted, so asking for the top five is immediate and sorts nothing at read time.

In [ ]:
r.delete("items:by_sales")

r.zadd("items:by_sales", {row["Item_ID"]: float(row["Item_AnnualSales"]) for row in items})

print("top 5 sellers")
for item_id, score in r.zrevrange("items:by_sales", 0, 4, withscores=True):
    print(f"  {item_id:<14} {int(score):>5}")

print()
print("bottom 3")
for item_id, score in r.zrange("items:by_sales", 0, 2, withscores=True):
    print(f"  {item_id:<14} {int(score):>5}")

In [ ]:
# everything that sold between 100 and 1000 units, in order
print(r.zrangebyscore("items:by_sales", 100, 1000, withscores=True))

print()
print("how many items at all:", r.zcard("items:by_sales"))
print("how many over 1000   :", r.zcount("items:by_sales", 1000, "+inf"))

This is the structure with no clean relational equivalent. `ORDER BY ... LIMIT 5` gives the same answer in SQL, but it sorts at query time, every time. Here the order is already there.

## Choosing a Structure

| Structure  | Holds                       | Reach for it when                   |
|------------|-----------------------------|-------------------------------------|
| String     | One value                   | Caching a value, counting           |
| Hash       | Field and value pairs       | Storing a record or object          |
| List       | Ordered, duplicates allowed | Queues, recent activity             |
| Set        | Unordered, unique           | Membership, set operations          |
| Sorted set | Unique, ordered by score    | Rankings, leaderboards, anything top N |

The structure gets chosen when the data is **written**, not when it is queried. That is the opposite of the relational habit, where one schema serves whatever query turns up later. So the design question in Redis is not how to model the data, but how it is going to be read.

Notice that the same item data went into a set *and* a sorted set above, for two different questions. Storing it twice is normal here, and would be a normalisation error in the MySQL half of the course.

## 6. Keys That Delete Themselves

Any key can be given a lifetime, after which Redis removes it without being asked.

In [ ]:
r.set("session:abc123", "C25")
r.expire("session:abc123", 60)              # seconds

print("ttl:", r.ttl("session:abc123"))

r.persist("session:abc123")                 # cancel the expiry
print("ttl after persist:", r.ttl("session:abc123"))

r.set("session:abc123", "C25", ex=60)       # set and expire in one call
print("ttl again:", r.ttl("session:abc123"))

Use the `ex=` form where you can. Setting the value and the lifetime in one call means there is no instant in between where the key exists with no expiry, which is a real failure mode if the process dies at the wrong moment.

`ttl` has two answers worth memorising, because both are easy to mistake for an error:

- `-1` means the key exists but has **no** expiry
- `-2` means the key is **gone** (or was never there)

In [ ]:
r.set("permanent:key", "no expiry on this one")

print("a key with no expiry:", r.ttl("permanent:key"))      # -1
print("a key that is gone  :", r.ttl("never:existed"))      # -2

In [ ]:
import time

r.set("short:lived", "watch this", ex=3)

for _ in range(5):
    print(f"ttl={r.ttl('short:lived'):>3}   value={r.get('short:lived')}")
    time.sleep(1)

Nothing in the SQL half of this course does that. Expiring a row in MySQL means writing a scheduled job that goes looking for stale rows and deletes them. Here the lifetime is a property of the key itself.

## 7. Redis in Front of a Database

This is the most common reason a team runs Redis alongside a relational database, and it brings together the string, the expiry, and the CAP discussion from Lecture 11.

Below stands in for a report that is expensive to compute. The `time.sleep` is doing the job of a slow query over a large table.

In [ ]:
def run_expensive_query():
    """Pretend this is a slow aggregate over a large orders table."""
    time.sleep(2)
    total = sum(float(row["Item_AnnualSales"]) for row in items)
    return str(total)

In [ ]:
def monthly_sales():
    key = "report:monthly_sales"
    cached = r.get(key)

    if cached is None:
        print("  cache miss, running the slow query")
        cached = run_expensive_query()
        r.set(key, cached, ex=300)        # keep the answer for 5 minutes
    else:
        print("  cache hit")

    return cached

In [ ]:
r.delete("report:monthly_sales")          # start from a cold cache

for attempt in range(1, 4):
    started = time.time()
    answer = monthly_sales()
    print(f"call {attempt}: {answer} in {time.time() - started:.3f}s")
    print()

The first call pays the two seconds. Every call for the next five minutes is answered from memory in well under a millisecond, and then the key expires and the cycle repeats.

This is the CAP discussion made concrete. The cached answer may be **up to five minutes stale**, which is a deliberate trade of consistency for speed. Choosing that window is a design decision, not a detail: five minutes may suit a sales dashboard and be completely wrong for an inventory count that someone is about to make a purchase against.

Two things to be careful about:

In [ ]:
# 1. Forget the expiry and the cache never refreshes. The report is now wrong forever.
r.set("report:no_expiry", run_expensive_query())
print("ttl:", r.ttl("report:no_expiry"), "<- -1, this will never update again")

# 2. An empty result caches as a value, but a failed query must not be cached at all.
#    Storing "" or None on failure means serving the failure for the next five minutes.
r.delete("report:no_expiry")
print("cleaned up")

## Tidying Up

Everything written by this notebook is still in database 0. Leave it if you want to poke at it; clear it with the cell below.

In [ ]:
print("keys before:", r.dbsize())

# r.flushdb()         # uncomment to empty the database completely

for key in ["page:views", "short:lived", "session:abc123",
            "permanent:key", "report:monthly_sales", "orders:pending"]:
    r.delete(key)

print("keys after :", r.dbsize())

## Points to Remember

- A value can be a structure, and the operations on it run on the server.
- Strings count atomically, which SQL needs a transaction and a row lock to match.
- Hashes are the nearest thing to a row, with none of the schema and none of the protection.
- Lists give you a queue; sets give you membership and set algebra without a join; sorted sets give you rankings that are never sorted at read time.
- Pick the structure for how the data will be **read**, and store it more than once if it will be read more than one way.
- Expiry is a property of the key, and the cache-with-a-TTL pattern is the CAP trade-off written in four lines of Python.